In [164]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers import MultiQueryRetriever
from dotenv import load_dotenv

load_dotenv()

True

In [165]:
youtube_video_id = "PHe0bXAIuk0"

ytt_api = YouTubeTranscriptApi()

fetched_transcript = ytt_api.fetch(
    youtube_video_id, 
    languages=["en"]
)

# print(fetched_transcript.snippets)
transcript = " ".join(item.text for item in fetched_transcript.snippets)

print(transcript)

  How the economic machine works, in 30 minutes. The economy works like a simple machine. But many people don't understand it — or they don't agree on how it works — and this has led to a lot of needless economic suffering. I feel a deep sense of responsibility to share my simple but practical economic template. Though it's unconventional, it has helped me to anticipate 
and sidestep the global financial crisis, and has worked well for me for over 30 years. Let's begin. Though the economy might seem complex, 
it works in a simple, mechanical way. It's made up of a few simple parts and a lot of simple transactions that are repeated over and over again a zillion times. These transactions are above all else driven by human nature, and they create 3 main forces that drive the economy. Number 1: Productivity growth Number 2: The Short term debt cycle and Number 3: The Long term debt cycle We'll look at these three forces
and how laying them on top of each other creates a good template for t

In [166]:
embedder = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-1')

In [167]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap = 800
)
docs = splitter.split_text(transcript,)
print(docs)
print(len(docs))

["How the economic machine works, in 30 minutes. The economy works like a simple machine. But many people don't understand it — or they don't agree on how it works — and this has led to a lot of needless economic suffering. I feel a deep sense of responsibility to share my simple but practical economic template. Though it's unconventional, it has helped me to anticipate \nand sidestep the global financial crisis, and has worked well for me for over 30 years. Let's begin. Though the economy might seem complex, \nit works in a simple, mechanical way. It's made up of a few simple parts and a lot of simple transactions that are repeated over and over again a zillion times. These transactions are above all else driven by human nature, and they create 3 main forces that drive the economy. Number 1: Productivity growth Number 2: The Short term debt cycle and Number 3: The Long term debt cycle We'll look at these three forces\nand how laying them on top of each other creates a good template fo

In [168]:
vector_store = Chroma.from_texts(
    texts=docs,
    embedding=embedder,
    collection_name='transcript'
)

GoogleGenerativeAIError: Error embedding content (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-embedding-1 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['aa968acd-0f63-4680-a00f-3504941b4902',
  '7bc3b944-9509-492c-b35e-fdb3f73bf196',
  '2e777af4-8e93-4fd8-b962-551549816010',
  '041e460e-66e2-4acc-8013-3f0d73b71b19',
  'ea63cb0e-7a83-4e62-a174-5af1aeef05ed',
  'efbfc168-461f-4054-867b-c4e9a2718571',
  '6f8086fc-a9dd-4eab-a0fc-d5dbfaeb3c1a',
  '353557e3-7e63-4264-97bb-9c74d15ca342',
  '76b67528-5287-4849-939b-aa2dd66a3969',
  'c9a6a056-24e1-46cd-9125-f2fa022da6fe',
  '1cf35d63-d961-4f69-a8a7-4c9479b57a99',
  'b5bb08bf-5096-4a1a-a0d8-cb35a69c6c97',
  'ffe9587c-5ba3-43bf-835c-cf61a205cb53',
  '97394137-bfe3-4e4a-9aa8-2ce9ee96cfb2',
  '77813609-9b08-4708-a72e-f8c425dcf74d',
  '0f2446c9-c249-4b78-82ce-8972ce0453c3',
  'baeb867c-ee21-4c0c-ba25-a0cf75afdb4e',
  '8a9d36e3-c83f-45c8-adfa-2ce490c7d3b2',
  '36816ed2-7288-44e4-af17-db8a6e6bcffd',
  '2a54f970-9d48-4b08-bb73-99e8c21bb8cc',
  'de076e37-8988-49bf-93e1-e9ff567b374a',
  '97586e11-052c-4bef-acc3-cd31b910e13a',
  '763b947b-f502-4d55-a677-15caf4d13123',
  '0d006f08-b096-4af8-a166-

In [ ]:
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k':5, 'lambda_mult':0.5}
)


In [ ]:
sub_query_model = ChatGoogleGenerativeAI(model = 'gemini-3.5-flash-lite')
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=sub_query_model
)

In [ ]:
compression_model = ChatGoogleGenerativeAI(model = 'gemini-3.5-flash-lite')
compressor = LLMChainExtractor.from_llm(llm=compression_model)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=multi_query_retriever
)

In [ ]:
model = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite')

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
def mergeDocuments(docs):
    print(docs)
    print(len(docs))
    return '\n\n'.join(item.page_content for item in docs)

In [ ]:
parser = StrOutputParser()

In [ ]:
parallel_chain = RunnableParallel(
    {"context" : retriever | RunnableLambda(mergeDocuments) | parser,
    "question" : RunnablePassthrough()
    }
)

In [ ]:
query = "what is economy"
final_chain = parallel_chain | prompt | model | parser

res = final_chain.invoke(query)

print(res)

[Document(metadata={}, page_content="How the economic machine works, in 30 minutes. The economy works like a simple machine. But many people don't understand it — or they don't agree on how it works — and this has led to a lot of needless economic suffering. I feel a deep sense of responsibility to share my simple but practical economic template. Though it's unconventional, it has helped me to anticipate \nand sidestep the global financial crisis, and has worked well for me for over 30 years. Let's begin. Though the economy might seem complex, \nit works in a simple, mechanical way. It's made up of a few simple parts and a lot of simple transactions that are repeated over and over again a zillion times. These transactions are above all else driven by human nature, and they create 3 main forces that drive the economy. Number 1: Productivity growth Number 2: The Short term debt cycle and Number 3: The Long term debt cycle We'll look at these three forces"), Document(metadata={}, page_con

GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 36.376746029s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '